In [16]:
from andromeda.core.agent import Agent
from andromeda.config import AgentConfig, ModelConfig
from andromeda.core.workflow import WorkflowBuilder
from pydantic import BaseModel
import sys
import tempfile
import subprocess
import re

In [17]:
modelconfig = ModelConfig(name='llama3.2:3b', provider='ollama', temperature=0.4)

In [18]:
class ReviewState(BaseModel):
    file_path: str
    code: str
    bug_review: str
    security_review: str
    quality_review: str
    execution_result: str
    error_details: str
    runtime_review: str
    final_report: str

In [19]:
def load_code(state):
    with open(state["file_path"], "r") as file:
        code = file.read()
    return {"code": code}

In [ ]:
bugAgent_prompt = f"""
    You are a senior Python debugging expert.
    Analyze this Python code.
    Find:
    - logical mistakes
    - runtime risks
    - missing validations
    - incorrect assumptions

    For every issue provide:
    Problem:
    Severity:
    Explanation:
    Fix:

    Code:{state["task"]}
"""

security_prompt = f"""
    You are` a Python security engineer.
    Review this code for security problems.
    Look for:
    - hardcoded passwords
    - API keys
    - unsafe functions
    - injection risks
    - insecure practices

    Provide:
    Issue:
    Severity:
    Explanation:
    Recommendation:
    """

In [21]:
bugAgent = Agent(
    AgentConfig(
        name = "bugAgent",
        model = modelconfig,
        prompt=bugAgent_prompt
    )
)

In [22]:
securityAgent = Agent(
    AgentConfig(
        name = "securityAgent",
        model = modelconfig,
        prompt=security_prompt
    )
)

In [ ]:
workflow = WorkflowBuilder(name = "wORKFLOW")
# (
# workflow
#     .start("first").run(load_code)
#     .then('second').run(bugAgent)
#     .finish('third').run(securityAgent)
# )

(
    workflow
    .start("research")
        .run(lambda state: {
            "research": load_code.run(state["query"])
        })

    .then("analysis")
        .run(lambda state: {
            "analysis": bugAgent.run({
                "query": state["query"],
                "research": state["research"],
            })
        })

    .finish("report")
        .run(lambda state: {
            "report": securityAgent.run({
                "research": state["research"],
                "analysis": state["analysis"],
            })
        })
)

In [24]:
response = workflow.execute(
    state={"file_path":"/home/rakeshchanda/TrainingFolder/CodingLive/AI_Code_Reviewer_Agent/samples/buggy_code.py"}
)

WorkflowExecutionError: <andromeda.core.agent.Agent object at 0x705f7b5d3320> is not a callable object

In [ ]:
print(response)